# M4A → Text Transcriber

Transcribes an `.m4a` file (up to 60 minutes) to text using OpenAI Whisper.

**How to use** (no terminal required):
1. Open this notebook in [Google Colab](https://colab.research.google.com/) (`File → Upload notebook`).
2. Click **Runtime → Run all**.
3. Paste your OpenAI API key when prompted (input is hidden).
4. Click **Choose Files** and pick your `.m4a`.
5. Wait — the transcript prints below and auto-downloads as a `.txt`.

Files larger than 25 MB are auto-split into 10-minute chunks with ffmpeg, then stitched back together.

In [ ]:
# 1. Install dependencies (ffmpeg ships pre-installed on Colab; this is a no-op there).
!apt-get -qq install -y ffmpeg > /dev/null
!pip -q install --upgrade openai

In [ ]:
# 2. Enter your OpenAI API key (hidden input).
import getpass, os
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

In [ ]:
# 3. Upload your .m4a file.
from google.colab import files
uploaded = files.upload()
input_path = next(iter(uploaded))
print(f"Uploaded: {input_path} ({len(uploaded[input_path]) / 1e6:.1f} MB)")

In [ ]:
# 4. Transcribe.
import subprocess, tempfile
from pathlib import Path
from openai import OpenAI

WHISPER_API_LIMIT_BYTES = 25 * 1024 * 1024
CHUNK_SECONDS = 600
MAX_DURATION_SECONDS = 60 * 60
MODEL = "whisper-1"  # change to "gpt-4o-transcribe" for higher quality
LANGUAGE = None       # e.g. "en" to skip auto-detection

def get_duration(path: Path) -> float:
    out = subprocess.check_output([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path),
    ])
    return float(out.strip())

def split_audio(src: Path, out_dir: Path, chunk_seconds: int):
    pattern = out_dir / "chunk_%03d.m4a"
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", str(src),
        "-f", "segment", "-segment_time", str(chunk_seconds),
        "-c", "copy", str(pattern),
    ], check=True)
    return sorted(out_dir.glob("chunk_*.m4a"))

def transcribe_file(client, path: Path) -> str:
    with path.open("rb") as f:
        kwargs = {"model": MODEL, "file": f}
        if LANGUAGE:
            kwargs["language"] = LANGUAGE
        return client.audio.transcriptions.create(**kwargs).text

src = Path(input_path)
duration = get_duration(src)
if duration > MAX_DURATION_SECONDS:
    raise SystemExit(f"audio is {duration/60:.1f} min; max supported is 60 min")
print(f"Duration: {duration/60:.1f} min, size: {src.stat().st_size / 1e6:.1f} MB")

client = OpenAI()
pieces = []
if src.stat().st_size <= WHISPER_API_LIMIT_BYTES and duration <= CHUNK_SECONDS:
    print("Transcribing whole file…")
    pieces.append(transcribe_file(client, src))
else:
    with tempfile.TemporaryDirectory() as tmp:
        tmp_dir = Path(tmp)
        print(f"Splitting into ~{CHUNK_SECONDS // 60}-min chunks…")
        chunks = split_audio(src, tmp_dir, CHUNK_SECONDS)
        for i, chunk in enumerate(chunks, 1):
            print(f"Transcribing chunk {i}/{len(chunks)}…")
            pieces.append(transcribe_file(client, chunk))

transcript = "\n\n".join(p.strip() for p in pieces if p.strip())
out_path = Path(src.stem + ".txt")
out_path.write_text(transcript + "\n", encoding="utf-8")
print(f"\nWrote {out_path} ({len(transcript)} chars)\n")
print("--- TRANSCRIPT ---\n")
print(transcript)

In [ ]:
# 5. Download the transcript to your computer.
from google.colab import files
files.download(str(out_path))